In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageOps
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torchvision.transforms as transforms

# ----------------------------------------------------
# 步驟 1：定義 Pipeline A 資料集 (維持影像長寬比)
# ----------------------------------------------------
class BreastDenseDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')
        
        # Pipeline A：補黑邊 (Padding)，避免腫瘤組織在形變中失去病理特徵
        w, h = img.size
        max_side = max(w, h)
        padding = ((max_side - w) // 2, (max_side - h) // 2, 
                   (max_side - w) - (max_side - w) // 2, (max_side - h) - (max_side - h) // 2)
        img = ImageOps.expand(img, padding, fill=0)
        img = img.resize((224, 224))
        
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
            
        return img, label

In [9]:
# ----------------------------------------------------
# 步驟 2：資料庫檔案搜尋與交叉驗證分割
# ----------------------------------------------------
path = "C:/Users/謝秉岑/.cache/kagglehub/datasets/dynemiesizumaki/cbis-ddsmpng/versions/1"

all_image_paths = []
all_labels = []

print("正在掃描硬碟影像檔案...")
for root, dirs, files in os.walk(path):
    if "roi_cropped.png" in files:
        # 良性 = 0, 惡性 = 1
        label = 1 if "MALIGNANT" in root.upper() else 0
        all_image_paths.append(os.path.join(root, "roi_cropped.png"))
        all_labels.append(label)

# 分割資料：80% 訓練與驗證, 20% 最終獨立測試集
train_paths, test_paths, train_labels, test_labels = train_test_split(
    all_image_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)
# 再從中切出 10% 作為訓練中的模擬考（驗證集）
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.1, stratify=train_labels, random_state=42
)

# ImageNet 規格的標準化轉換
data_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 建立高效資料批次產生器
train_loader = DataLoader(BreastDenseDataset(train_paths, train_labels, data_transforms), batch_size=32, shuffle=True)
val_loader = DataLoader(BreastDenseDataset(val_paths, val_labels, data_transforms), batch_size=32, shuffle=False)
test_loader = DataLoader(BreastDenseDataset(test_paths, test_labels, data_transforms), batch_size=32, shuffle=False)

正在掃描硬碟影像檔案...


In [10]:
# ----------------------------------------------------
# 步驟 3：發動 🔧 DenseNet121 與微調引導設定
# ----------------------------------------------------
from torchvision.models import densenet121, DenseNet121_Weights

print("正在載入官方預訓練 DenseNet121 權重...")
weights = DenseNet121_Weights.DEFAULT
model = densenet121(weights=weights)

# 遷移學習策略：先凍結全面神經元
for param in model.parameters():
    param.requires_grad = False

# 解凍微軟 AI 論文最推薦微調的後段特徵區塊：denseblock4 與 transition 層
# 讓深層結構的「特徵重用機制」去適應乳房攝影的特異紋理
for param in model.features.denseblock4.parameters():
    param.requires_grad = True
for param in model.features.norm5.parameters():
    param.requires_grad = True

# 置換官方的 classifier（將原本的 1000 類輸出，修改為臨床二分類）
num_ftrs = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.4), # 引入 Dropout 機制強效防止過擬合
    nn.Linear(256, 1) # 輸出 1 個對數機率值，後續交由 BCEWithLogitsLoss 處理
)

# GPU 加速運算配給
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

正在載入官方預訓練 DenseNet121 權重...


In [11]:
# ----------------------------------------------------
# 步驟 4：定義微調優化器與損失函數
# ----------------------------------------------------
criterion = nn.BCEWithLogitsLoss()
# 僅將需要更新梯度（requires_grad=True）的參數送入 Adam 優化器，並採用 1e-5 低學習率
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

# ----------------------------------------------------
# 步驟 5：動態計算圖訓練迴圈 (精準迭代 30 輪)
# ----------------------------------------------------
epochs = 30
print(f"硬體核心已就緒：[{device}]。啟動 DenseNet121 醫療篩檢模型訓練...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward() # 手動觸發反向傳播
        optimizer.step() # 更新權重步伐
        
        running_loss += loss.item() * inputs.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs} - 訓練損失 (Train Loss): {epoch_loss:.4f}")

硬體核心已就緒：[cpu]。啟動 DenseNet121 醫療篩檢模型訓練...
Epoch 1/30 - 訓練損失 (Train Loss): 0.6903
Epoch 2/30 - 訓練損失 (Train Loss): 0.6770
Epoch 3/30 - 訓練損失 (Train Loss): 0.6689
Epoch 4/30 - 訓練損失 (Train Loss): 0.6652
Epoch 5/30 - 訓練損失 (Train Loss): 0.6668
Epoch 6/30 - 訓練損失 (Train Loss): 0.6540
Epoch 7/30 - 訓練損失 (Train Loss): 0.6557
Epoch 8/30 - 訓練損失 (Train Loss): 0.6553
Epoch 9/30 - 訓練損失 (Train Loss): 0.6388
Epoch 10/30 - 訓練損失 (Train Loss): 0.6399
Epoch 11/30 - 訓練損失 (Train Loss): 0.6356
Epoch 12/30 - 訓練損失 (Train Loss): 0.6179
Epoch 13/30 - 訓練損失 (Train Loss): 0.6246
Epoch 14/30 - 訓練損失 (Train Loss): 0.6151
Epoch 15/30 - 訓練損失 (Train Loss): 0.6214
Epoch 16/30 - 訓練損失 (Train Loss): 0.6053
Epoch 17/30 - 訓練損失 (Train Loss): 0.6111
Epoch 18/30 - 訓練損失 (Train Loss): 0.6044
Epoch 19/30 - 訓練損失 (Train Loss): 0.5907
Epoch 20/30 - 訓練損失 (Train Loss): 0.5930
Epoch 21/30 - 訓練損失 (Train Loss): 0.5984
Epoch 22/30 - 訓練損失 (Train Loss): 0.5809
Epoch 23/30 - 訓練損失 (Train Loss): 0.5676
Epoch 24/30 - 訓練損失 (Train Loss): 0.5778
Epoch 25

In [17]:
# ----------------------------------------------------
# 步驟 6：獨立測試集最終驗收（已加入 Threshold 設定）
# ----------------------------------------------------
model.eval()
all_preds = []
all_targets = []

# 【這裡就是妳的設定區】
# 妳可以隨時把 0.25 改成 0.2 或 0.3 來測試
THRESHOLD = 0.42 

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze()
        
        # 使用設定好的門檻進行判定
        preds = (torch.sigmoid(outputs) > THRESHOLD).cpu().numpy().astype(int)
        all_preds.extend(preds)
        all_targets.extend(labels.numpy().astype(int))

print("\n" + "="*45)
print(f"【調整門檻至 {THRESHOLD}：DenseNet121 測試集報表】")
print(classification_report(all_targets, all_preds, target_names=['Benign', 'Malignant']))
print("="*45)


【調整門檻至 0.42：DenseNet121 測試集報表】
              precision    recall  f1-score   support

      Benign       0.66      0.65      0.65        57
   Malignant       0.53      0.55      0.54        42

    accuracy                           0.61        99
   macro avg       0.60      0.60      0.60        99
weighted avg       0.61      0.61      0.61        99

